# Reaction prediction

This notebook helps to understand how reaction outcomes can be predicted over one step. The dataset in use is the training set of the USPTO-MIT (409k; since sampling is needed anyway, npo need for the validation or test set) as obtained here: https://github.com/CC-SXF/DataSet-USPTO 

Sampling was introduced in order to limit the computation time.

1) Load dependencies

In [ ]:

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from tqdm import tqdm


2) Load USPTO-MIT data

In [ ]:

inputs_df = pd.read_csv(
    "uspto_mit_inputs.txt",
    header=None,
    names=["inputs"]
)


targets_df = pd.read_csv(
    "uspto_mit_targets.txt",
    header=None,
    names=["targets"]
)


# Combine into a single DataFrame
df = pd.concat([inputs_df, targets_df], axis=1)


print("Number of reactions:", len(df))
df.head()

In [ ]:
df = df.sample(100000, replace=True, random_state=42)
print(len(df))
df.head()

In [ ]:
# Clean whitespaces
df["inputs"] = df["inputs"].str.replace(" ", "", regex=False)

df["targets"] = df["targets"].str.replace(" ", "", regex=False)
df.head()

3) Reaction featurisation

Combine reactants and reagents, ignore conditions, use MorganFPs.

In [ ]:
# helper function 
def smiles_to_fp(smiles, n_bits=2048, radius=2):
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    fp = fpgen.GetFingerprint(mol)
    # return fp
    fp_arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)
    return fp_arr



In [ ]:
X = []
y = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    rxn_smiles = str(row["inputs"])
    fp = smiles_to_fp(rxn_smiles)
    
    if fp is None:
        continue
    
    X.append(fp)
    y.append(row["targets"])

X = np.array(X)
y = np.array(y)

print("Usable reactions:", len(X))


4) Convert Products to classification labels

We treat the prediction as multiclass prediction. This assumes that there is **only one product** for one reaction - which is mostly not true for reality. 

In [ ]:
product_to_id = {p: i for i, p in enumerate(sorted(set(y)))}
id_to_product = {i: p for p, i in product_to_id.items()}

y_ids = np.array([product_to_id[p] for p in y])

print("Unique products:", len(product_to_id))

5) Train/test split and optional sampling

In [ ]:
MAX_SAMPLES = 20000  # set None to use full dataset

if MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X = X[idx]
    y_ids = y_ids[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_ids, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

6) Define and train ML model.

Using a probabilistic model (e.g. as here: RF) allows us to use effciently evaluation by top-k results and makes interpretation easy.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_leaf=10,
    min_samples_split=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

7) Top-k accuracy determination

As more reaction paths and therefor multiple different products are possible, looking at one reaction outcome is not very useful.

Instead we are looking at the top-k results for the evaluation. 

In [ ]:
def top_k_accuracy(y_true, probs, k):
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return np.mean([
        y_true[i] in top_k[i]
        for i in range(len(y_true))
    ])

In [ ]:
probs = model.predict_proba(X_test)

for k in [5, 10, 20, 50]:
    acc = top_k_accuracy(y_test, probs, k)
    print(f"Top-{k} accuracy: {acc:.3f}")

In [ ]:
i = np.random.randint(len(X_test))

true_product = id_to_product[y_test[i]]
top_preds = np.argsort(probs[i])[-5:][::-1]

print("True product:")
print(true_product)

print("\nTop-5 predicted products:")
for pid in top_preds:
    print("-", id_to_product[pid])


9) Discussion

- How would other tasks be done (e.g. yield prediction)?
- Why are the top-k accuracies super low?
- What could be done to improve this predictor?
- What experimental information is missing?
- How would this model be used inside a synthesis planner?

